# 4. Generación de indicadores — autónoma

Cuaderno para producir las salidas de indicadores: resúmenes, conteos, siete Parquet enriquecidos y el CSV de marcadores.

Antes de calcular, **lee siempre las versiones `_autonomo` de RNPDNO, BANCI y PUI** y pide al pipeline que valide/actualice su snapshot. Si los insumos no cambiaron, reutiliza el caché vigente; si cambiaron, reconstruye matching y derivadas automáticamente. El catálogo INEGI de municipios también se lee localmente.

## 1. Imports y configuración

In [ ]:
from __future__ import annotations

import json
import sys
from dataclasses import dataclass, replace
from datetime import datetime
from pathlib import Path
from typing import Iterable, Optional

import numpy as np
import pandas as pd
from IPython.display import display


def encontrar_raiz(inicio: Path) -> Path:
    """Busca el repositorio en el directorio actual y sus padres."""
    for candidato in (inicio.resolve(), *inicio.resolve().parents):
        if ((candidato / "output" / "cache" / "universo").is_dir()
                and (candidato / "input" / "catalogos_inegi").is_dir()
                and (candidato / "config.py").is_file()):
            return candidato
    raise FileNotFoundError("No se encontró raíz del proyecto con config.py, caché y catálogo INEGI.")


RAIZ = encontrar_raiz(Path.cwd())
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

# Usa el pipeline con caché inteligente: revisa firmas de los tres Parquet
# autónomos y reconstruye el universo si cambió cualquiera de sus insumos.
import config
from src.settings import Settings as PipelineSettings
from src.pipeline.universo import Universo as PipelineUniverso

PERFIL_PIPELINE = "reporte_diario"
_cfg_pipeline = PipelineSettings.desde_perfil(PERFIL_PIPELINE)
_rutas_etl = {
    "RNPDNO": _cfg_pipeline.rnpdno_etl_parquet,
    "BANCI": _cfg_pipeline.banci_etl_parquet,
    "PUI": _cfg_pipeline.confrontas_file,
}
for _nombre, _ruta in _rutas_etl.items():
    if not _ruta.is_file():
        raise FileNotFoundError(f"Falta el Parquet autónomo de {_nombre}: {_ruta}")
    if not _ruta.stem.endswith("_autonomo"):
        raise ValueError(f"La entrada de {_nombre} no es la versión autónoma: {_ruta}")

# La clave no se fija a mano: se calcula desde el perfil y las entradas vigentes.
CLAVE_SNAPSHOT = _cfg_pipeline.clave_universo
SNAPSHOT_DIR = _cfg_pipeline.cache_dir / "universo" / CLAVE_SNAPSHOT
RESULTADOS_DIR = _cfg_pipeline.resultados_dir
CATALOGO_MUNICIPIOS_FILE = config.CATALOGO_MUNICIPIOS_FILE
INDICADORES_FILE = RESULTADOS_DIR / "indicadores_selectos_desaparecidos.csv"

ENTIDADES = {
    1:"Aguascalientes",2:"Baja California",3:"Baja California Sur",4:"Campeche",5:"Coahuila",6:"Colima",
    7:"Chiapas",8:"Chihuahua",9:"Ciudad de México",10:"Durango",11:"Guanajuato",12:"Guerrero",
    13:"Hidalgo",14:"Jalisco",15:"México",16:"Michoacán",17:"Morelos",18:"Nayarit",19:"Nuevo León",
    20:"Oaxaca",21:"Puebla",22:"Querétaro",23:"Quintana Roo",24:"San Luis Potosí",25:"Sinaloa",
    26:"Sonora",27:"Tabasco",28:"Tamaulipas",29:"Tlaxcala",30:"Veracruz",31:"Yucatán",32:"Zacatecas",
    33:"Nacido en el Extranjero",
}
PERIODOS_TIEMPO = {
    "min-max":(None,None),"min-01ene06":(None,"2006-01-01"),"01ene06-max":("2006-01-01",None),
    "min-01oct24":(None,"2024-10-01"),"01oct24-max":("2024-10-01",None),"01ago25-max":("2025-08-01",None),
    "01oct24-01ago25":("2024-10-01","2025-08-01"),"01ago26-max":("2026-08-01",None),
    "01jul26-max":("2026-07-01",None),
}
BASES = {
    "rn":("rn",None),"ba":("ba",None),
    "rn-no-ba":("rn",("in_banci_derivada_rnpdno_banci",False)),
    "rn-si-ba":("rn",("in_banci_derivada_rnpdno_banci",True)),
    "ba-no-rn":("ba",("in_rnpdno_derivada_rnpdno_banci",False)),
    "ba-si-rn":("ba",("in_rnpdno_derivada_rnpdno_banci",True)),
}
DIMENSIONES = {
    "rn":{"fecha":"fecha_hechos_modificada_derivada_rnpdno","cve_ent":"idestado_derivada_rnpdno",
          "entidad":"nombre_entidad_derivada_rnpdno","estatus_loc":"estatus_loc_derivada_rnpdno"},
    "ba":{"fecha":"fha_de_hchos_derivada_banci","cve_ent":"cve_entidad_derivada_banci",
          "entidad":"nombre_entidad_derivada_banci","estatus_loc":"estatus_loc_derivada_banci"},
}
NEGACIONES={"sdel":"cdel","sind":"cind","scurp":"ccurp","cisf":"sisf","sfvac":"cfvac","sfval":"cfval"}
CONSTANTES={"rn":{},"ba":{"cind":True,"cfval":True,"cfvac":False,"fis":True,"com":False,"por":False}}
BANDERAS_ETL={"rn":{"ccurp":"curp_vic_valida_derivada_rnpdno"},"ba":{"ccurp":"curp_valida_derivada_banci"}}
ESTATUS={"all","des","loc"}

print(f"Raíz: {RAIZ}")
print(f"Perfil de datos: {PERFIL_PIPELINE}")
print("Entradas que se verifican en cada ejecución:")
for _nombre, _ruta in _rutas_etl.items():
    print(f"  {_nombre}: {_ruta}")
print(f"Snapshot objetivo: {SNAPSHOT_DIR}")
print(f"Salida de indicadores: {RESULTADOS_DIR}")

## 2. Funciones locales: rebanadas, universo y exportación

In [ ]:
@dataclass(frozen=True, order=True)
class Slice:
    """Objeto inmutable que declara una rebanada y su nombre canónico."""
    base: str = "rn"
    periodo: str = "min-max"
    estatus: str = "all"
    subfiltro: str = "all"
    estado: Optional[int] = None

    def __post_init__(self):
        if self.base not in BASES:
            raise ValueError(f"Base '{self.base}' desconocida: {sorted(BASES)}")
        if self.estatus not in ESTATUS:
            raise ValueError(f"Estatus '{self.estatus}' desconocido: {sorted(ESTATUS)}")
        if self.periodo not in PERIODOS_TIEMPO:
            raise ValueError(f"Período '{self.periodo}' desconocido: {sorted(PERIODOS_TIEMPO)}")
        if self.estado is not None and self.estado not in ENTIDADES:
            raise ValueError(f"Entidad {self.estado} desconocida")

    @property
    def nombre(self):
        partes=["reg",self.base,self.periodo]
        if self.estatus!="all":partes.append(self.estatus)
        if self.subfiltro!="all":partes.append(self.subfiltro)
        if self.estado is not None:partes.append(f"ent{self.estado:02d}")
        return "_".join(partes)

    @property
    def entidad_texto(self):
        return ENTIDADES.get(self.estado,"Nacional") if self.estado else "Nacional"

    @property
    def padre(self):
        if self.estado is not None:return replace(self,estado=None)
        if self.subfiltro!="all":
            tokens=self.subfiltro.split("_")[:-1]
            return replace(self,subfiltro="_".join(tokens) if tokens else "all")
        if self.estatus!="all":return replace(self,estatus="all")
        return replace(self,periodo="min-max")

    def con_(self,**cambios):return replace(self,**cambios)
    def por_entidad(self,cve):return self.con_(estado=cve)


def expandir_entidades(base, entidades=None):
    """Crea una rebanada por clave de entidad."""
    claves=entidades if entidades is not None else sorted(ENTIDADES)
    return [base.por_entidad(c) for c in claves]


def columna_bandera(df, token):
    cols=[c for c in df.columns if c.startswith(f"is_{token}_derivada_")]
    if len(cols)!=1:raise KeyError(f"Bandera {token}: columnas encontradas {cols}")
    return cols[0]


class Universo:
    """Carga snapshot nacional y aplica las mismas máscaras de slice que el pipeline."""
    def __init__(self,rn,ba,meta,auditoria=None):
        self.rn,self.ba,self.meta,self.auditoria=rn,ba,meta,auditoria
        self._cache_periodos={}

    @classmethod
    def cargar(cls,directorio):
        esperados=[directorio/n for n in ("rn.parquet","ba.parquet","meta.json")]
        faltan=[str(p) for p in esperados if not p.is_file()]
        if faltan:raise FileNotFoundError("Snapshot incompleto: "+", ".join(faltan))
        rn=pd.read_parquet(esperados[0]);ba=pd.read_parquet(esperados[1])
        meta=json.loads(esperados[2].read_text(encoding="utf-8"))
        if len(rn)!=meta.get("n_rn") or len(ba)!=meta.get("n_ba"):
            raise ValueError("Las filas del snapshot no coinciden con meta.json")
        ap=directorio/"auditoria.csv"
        auditoria=pd.read_csv(ap) if ap.is_file() else None
        return cls(rn,ba,meta,auditoria)

    @staticmethod
    def _frame(s):return BASES[s.base][0] if isinstance(s,Slice) else s

    def columna(self,s,dimension):return DIMENSIONES[self._frame(s)][dimension]

    def _col_bandera(self,frame,df,token):
        col=BANDERAS_ETL[frame].get(token)
        if col is not None:
            if col not in df.columns:raise KeyError(col)
            return col
        return columna_bandera(df,token)

    def columna_bandera(self,s,token):
        frame=self._frame(s);df=self.rn if frame=="rn" else self.ba
        return self._col_bandera(frame,df,token)

    def _mascara_periodo(self,frame,periodo):
        llave=(frame,periodo)
        if llave in self._cache_periodos:return self._cache_periodos[llave]
        df=self.rn if frame=="rn" else self.ba
        fecha=df[self.columna(frame,"fecha")]
        ini,fin=PERIODOS_TIEMPO[periodo]
        mask=np.ones(len(df),dtype=bool)
        nulas=fecha.isna().to_numpy()
        if ini is not None:mask &= (fecha>=pd.to_datetime(ini)).to_numpy()|nulas
        if fin is not None:mask &= (fecha<pd.to_datetime(fin)).to_numpy()|nulas
        self._cache_periodos[llave]=mask
        return mask

    def _bandera(self,frame,df,token,s):
        base=NEGACIONES.get(token,token)
        if base in CONSTANTES[frame]:
            valores=np.full(len(df),CONSTANTES[frame][base],dtype=bool)
        else:
            valores=df[self._col_bandera(frame,df,base)].fillna(False).astype(bool).to_numpy()
        return ~valores if token in NEGACIONES else valores

    def _frame_y_mascara(self,s):
        frame,pertenencia=BASES[s.base]
        df=self.rn if frame=="rn" else self.ba
        mask=self._mascara_periodo(frame,s.periodo).copy()
        if pertenencia is not None:
            col,valor=pertenencia
            b=df[col].fillna(False).to_numpy()
            mask &= b if valor else ~b
        if s.estatus!="all":
            mask &= df[self.columna(s,"estatus_loc")].eq(s.estatus).fillna(False).to_numpy()
        if s.subfiltro!="all":
            for token in s.subfiltro.split("_"):
                mask &= self._bandera(frame,df,token,s)
        if s.estado is not None:
            mask &= df[self.columna(s,"cve_ent")].eq(s.estado).fillna(False).to_numpy()
        return df,mask
    def contar(self,s):return int(self._frame_y_mascara(s)[1].sum())

    def slice(self,s):
        df,mask=self._frame_y_mascara(s)
        return df[mask]

    def columna_frame(self,s):return BASES[s.base][0] if isinstance(s,Slice) else s

    def rango_fechas(self,s):
        frame=self.columna_frame(s);ini,fin=PERIODOS_TIEMPO[s.periodo]
        fechas=(self.rn if frame=="rn" else self.ba)[self.columna(frame,"fecha")]
        f_min,f_max=fechas.min(),fechas.max()
        desde=f_min if ini is None else max(pd.Timestamp(ini),f_min)
        hasta=f_max if fin is None else min(pd.Timestamp(fin)-pd.Timedelta(days=1),f_max)
        return desde,hasta

    def indicadores(self,slices,extra=None):
        salida={s.nombre:self.contar(s) for s in slices}
        salida["entidad_texto"]="Nacional"
        fecha_rn=self.meta.get("fecha_corte_rnpdno");fecha_ba=self.meta.get("fecha_corte_banci")
        for base,fecha in (("rn",fecha_rn),("ba",fecha_ba)):
            for partes in ("dia_mes_anio","mes_anio","anio"):
                salida[f"fha_{base}_{partes}"]=formatear_fecha_es(fecha,partes) if fecha else "Sin Fecha"
        hoy=datetime.now().strftime("%Y-%m-%d")
        for partes in ("dia_mes_anio","dia_mes","mes_anio","anio"):
            salida[f"fha_hoy_{partes}"]=formatear_fecha_es(hoy,partes)
        if extra:salida.update(extra)
        return salida

    def exportar_indicadores(self,slices,destino,extra=None):
        valores=self.indicadores(slices,extra=extra)
        df=pd.DataFrame([{"Variable":k,"Valor":v} for k,v in valores.items()])
        destino.parent.mkdir(parents=True,exist_ok=True)
        df.to_csv(destino,index=False,encoding="utf-8-sig")
        print(f"📊 {len(df)} indicadores → {destino}")
        return df

    def resumen(self):
        return pd.DataFrame([{"base":nombre,"registros":self.contar(Slice(nombre))} for nombre in BASES])


def formatear_fecha_es(fecha,partes="dia_mes_anio"):
    """Replica el formato de fecha español usado por el pipeline original."""
    if not fecha or not isinstance(fecha,str):return fecha
    trozos=fecha.split("-")
    if len(trozos)!=3:return fecha
    anio,mes,dia=trozos
    meses={"01":"enero","02":"febrero","03":"marzo","04":"abril","05":"mayo","06":"junio","07":"julio","08":"agosto","09":"septiembre","10":"octubre","11":"noviembre","12":"diciembre"}
    valores={"dia":str(int(dia)),"mes":meses.get(mes,mes),"anio":anio}
    return " de ".join(valores[t] for t in partes.split("_"))


def con_claves_geo(df):
    """Añade cve_mun, cvegeo y nombre_municipio al final, con nulos si faltan claves."""
    col_mun=next((c for c in ("idmunicipio_derivada_rnpdno","id_mun_hchos_banci") if c in df.columns),None)
    if col_mun is None:raise KeyError(f"Sin columna de municipio. Columnas: {sorted(df.columns)}")
    sfx="derivada_rnpdno" if col_mun.endswith("_rnpdno") else "derivada_banci"
    col_ent="idestado_derivada_rnpdno" if sfx=="derivada_rnpdno" else "cve_entidad_derivada_banci"
    out=df.copy()
    ent=pd.to_numeric(out[col_ent],errors="coerce").astype("Int64")
    mun=pd.to_numeric(out[col_mun],errors="coerce").astype("Int64")
    completo=ent.notna() & mun.notna()
    cvegeo=pd.Series(pd.NA,index=out.index,dtype="string")
    cvegeo[completo]=(ent[completo].astype(int).map("{:02d}".format)+mun[completo].astype(int).map("{:03d}".format))
    out[f"cve_mun_{sfx}"]=mun
    out[f"cvegeo_{sfx}"]=cvegeo
    out[f"nombre_municipio_{sfx}"]=cvegeo.map(MAPA_MUNICIPIOS).astype("string")
    return out

print("Slice, Universo, geografía e indicadores están definidos dentro de este notebook.")

## 3. Preparar el universo actualizado y el catálogo municipal

Se validan los tres Parquet autónomos y se construye el universo con sus fechas y matching actuales. Los resultados se usan directamente desde memoria; el caché en disco es auxiliar y, si no puede escribirse (por ejemplo, por un kernel abierto antes de un cambio de código), no detiene la generación de indicadores.

In [ ]:
if not CATALOGO_MUNICIPIOS_FILE.is_file():
    raise FileNotFoundError(f"No existe catálogo INEGI: {CATALOGO_MUNICIPIOS_FILE}")

# Lee/valida los tres Parquet autónomos y solicita al pipeline el universo vigente.
# Si el caché de disco no puede guardarse, el objeto construido en memoria sigue
# siendo válido para esta ejecución; no hacemos depender los indicadores del meta.json.
u_pipeline = PipelineUniverso.construir(_cfg_pipeline, verbose=True)
CLAVE_SNAPSHOT = _cfg_pipeline.clave_universo
SNAPSHOT_DIR = _cfg_pipeline.cache_dir / "universo" / CLAVE_SNAPSHOT

_cat_mun = pd.read_csv(CATALOGO_MUNICIPIOS_FILE, dtype=str, encoding="utf-8")
requeridas_catalogo = {"CVEGEO", "NOM_MUN"}
if not requeridas_catalogo.issubset(_cat_mun.columns):
    raise KeyError(f"Al catálogo le faltan columnas {requeridas_catalogo-set(_cat_mun.columns)}")
MAPA_MUNICIPIOS = dict(zip(_cat_mun["CVEGEO"], _cat_mun["NOM_MUN"]))

# Usar el universo recién construido en memoria evita leer un snapshot viejo
# o fallar si el caché no se escribió por un kernel desactualizado.
u = Universo(
    rn=u_pipeline.rn.copy(),
    ba=u_pipeline.ba.copy(),
    meta=dict(u_pipeline.meta),
    auditoria=u_pipeline.auditoria,
)
if u.meta.get("perfil") != PERFIL_PIPELINE:
    raise ValueError(f"Perfil esperado {PERFIL_PIPELINE}; perfil obtenido={u.meta.get('perfil')!r}")
print(f"Snapshot/caché asociado: {SNAPSHOT_DIR}")
print(f"Universo calculado en esta ejecución: {u.meta.get('generado')}")
print(f"RNPDNO: {len(u.rn):,} | BANCI: {len(u.ba):,} | FUBs matcheados: {u.meta.get('n_fubs_matcheados', 0):,}")
print("\nResumen del universo vigente:")
print(u.resumen().to_string(index=False))
if u.auditoria is not None:
    print("\nAuditoría de matching:")
    print(u.auditoria.to_string(index=False))

## 4. Conteos, exploración y análisis de coincidencias

In [ ]:
REQUERIDOS = [
    Slice("rn", "01oct24-max"),
    Slice("rn", "01oct24-max", estatus="des"),
    Slice("rn-si-ba", "01oct24-max", estatus="des"),
]
print("Conteos de rebanadas requeridas:")
for s in REQUERIDOS:
    print(f"  {s.nombre:<40} {u.contar(s):>10,}")

df = u.slice(Slice("rn-si-ba", "01oct24-max", estatus="des"))
print("\nDimensiones de rn-si-ba sexenio/desaparecidos:", df.shape)
display(df.head())

s_match = Slice("rn-si-ba", "01oct24-max", estatus="des")
df_match = u.slice(s_match)
conteo_match = (
    df_match["tipo_match_derivada_rnpdno_banci"]
    .value_counts(dropna=False)
    .rename_axis("tipo_match")
    .reset_index(name="registros")
)
conteo_match = conteo_match[conteo_match["registros"] > 0]
conteo_match["pct"] = (100 * conteo_match["registros"] / len(df_match)).round(1)
n_fubs = df_match["fub_rnpdno"].nunique()
print(f"\n{s_match.nombre}: {len(df_match):,} registros | {n_fubs:,} FUBs únicos")
print(conteo_match.to_string(index=False))
print("\nCobertura por método (un FUB cuenta en cada método que lo sostiene):")
for metodo in ("estandar", "manual", "refinado"):
    n = df_match["tipo_match_derivada_rnpdno_banci"].str.contains(metodo, regex=False).sum()
    print(f"  {metodo:<12} {n:>6,}  ({100*n/len(df_match):.1f}%)")

## 5. Enriquecer y exportar las mismas siete rebanadas

In [ ]:
A_EXPORTAR = [
    Slice("rn", "min-max"),
    Slice("rn", "min-max", estatus="des"),
    Slice("rn", "min-max", estatus="loc"),
    Slice("rn", "01oct24-max", estatus="des"),
    Slice("rn-no-ba", "01oct24-max", estatus="des", subfiltro="sfval"),
    Slice("rn-no-ba", "01oct24-max", estatus="des", subfiltro="sfval_sfvac"),
    Slice("rn-si-ba", "01oct24-max", estatus="des"),
]
RESULTADOS_DIR.mkdir(parents=True, exist_ok=True)
exportados = {}
for s in A_EXPORTAR:
    df_s = con_claves_geo(u.slice(s))
    destino = RESULTADOS_DIR / f"{s.nombre}.parquet"
    df_s.to_parquet(destino, index=False, engine="pyarrow")
    exportados[s.nombre] = df_s
    print(f"💾 {s.nombre:<24} {len(df_s):>8,} registros | "
          f"{df_s['cvegeo_derivada_rnpdno'].isna().sum():>6,} sin municipio | "
          f"{df_s['fechalocaliza_derivada_rnpdno'].notna().sum():>7,} con fecha de localización → {destino.name}")

display(exportados["reg_rn_min-max_loc"][
    ["fub_rnpdno", "idestado_derivada_rnpdno", "nombre_entidad_derivada_rnpdno",
     "cve_mun_derivada_rnpdno", "cvegeo_derivada_rnpdno", "nombre_municipio_derivada_rnpdno",
     "fechacaptura_derivada_rnpdno", "fechalocaliza_derivada_rnpdno"]
].head())